Notebook 01 — Exploration et profilage des données===================================================Projet : Détection de biais dans les données de santé au travailAuteure : Laure Bonnefond — IPRP / IST — Master IA Data IPSSI BordeauxCe notebook charge les trois sources de données (AT/MP, DARES SUMER, INSEE)et réalise un profilage complet : types, valeurs manquantes, distributions.Sources :- AT/MP : Assurance Maladie — Risques professionnels (open data)- SUMER : DARES — Enquête Surveillance médicale des expositions (2017)- INSEE : Population active par secteur, âge, sexe

# 1. Configuration et imports


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Style graphique cohérent
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.grid': True,
    'grid.alpha': 0.3,
})

PALETTE = ['#0F6E56', '#185FA5', '#D85A30', '#534AB7', '#993556',
           '#BA7517', '#639922', '#E24B4A', '#5F5E5A']

print("Environnement configuré.")
print(f"pandas {pd.__version__} | numpy {np.__version__} | matplotlib {matplotlib.__version__}")


# 2. Chargement des données


## 2.1 Données AT/MP (Assurance Maladie)
Indicateurs de sinistralité par secteur d'activité (CTN A à I).
Source : Branche Risques Professionnels — Assurance Maladie.


In [ ]:
df_atmp = pd.read_csv('../data/raw/atmp_par_secteur.csv')
print(f"AT/MP : {df_atmp.shape[0]} lignes × {df_atmp.shape[1]} colonnes")
df_atmp


## 2.2 Données SUMER / DARES
Pourcentage de salariés exposés à chaque risque professionnel, par secteur.
Source : Enquête SUMER 2017 — DARES (Synthèse Stat n°35).


In [ ]:
df_sumer = pd.read_csv('../data/raw/sumer_expositions_secteur.csv')
print(f"SUMER : {df_sumer.shape[0]} lignes × {df_sumer.shape[1]} colonnes")
df_sumer


## 2.3 Données INSEE (référence population)
Répartition de la population active par secteur, sexe et tranche d'âge.
Source : INSEE — Enquête Emploi.


In [ ]:
df_insee = pd.read_csv('../data/raw/insee_population_active_secteur.csv')
print(f"INSEE : {df_insee.shape[0]} lignes × {df_insee.shape[1]} colonnes")
df_insee


# 3. Profilage des données


## 3.1 Types et valeurs manquantes


In [ ]:
def profiler(df, nom):
    """Affiche un profil synthétique d'un DataFrame."""
    print(f"\n{'='*60}")
    print(f"  PROFIL : {nom}")
    print(f"{'='*60}")
    print(f"  Dimensions : {df.shape[0]} lignes × {df.shape[1]} colonnes")
    print(f"  Mémoire    : {df.memory_usage(deep=True).sum() / 1024:.1f} Ko")
    print(f"\n  Types des colonnes :")
    for col in df.columns:
        n_null = df[col].isnull().sum()
        pct_null = n_null / len(df) * 100
        flag = f" ⚠️ {n_null} manquantes ({pct_null:.0f}%)" if n_null > 0 else ""
        print(f"    {col:40s} {str(df[col].dtype):10s}{flag}")
    
    # Stats descriptives pour les colonnes numériques
    numeriques = df.select_dtypes(include=[np.number])
    if not numeriques.empty:
        print(f"\n  Statistiques descriptives :")
        print(numeriques.describe().round(1).to_string())
    return None

profiler(df_atmp, "AT/MP — Sinistralité par secteur")


In [ ]:
profiler(df_sumer, "SUMER — Expositions par secteur")


In [ ]:
profiler(df_insee, "INSEE — Population active par secteur")


## 3.2 Contrôles de cohérence


In [ ]:
print("CONTRÔLES DE COHÉRENCE")
print("=" * 50)

# Vérifier que les % H/F font bien ~100%
for df, nom in [(df_atmp, "AT/MP"), (df_insee, "INSEE")]:
    col_h = [c for c in df.columns if 'hommes' in c.lower()][0]
    col_f = [c for c in df.columns if 'femmes' in c.lower()][0]
    total_hf = df[col_h] + df[col_f]
    ecart = (total_hf - 100).abs().max()
    status = "✅" if ecart < 1 else "⚠️"
    print(f"{status} {nom} — Total H+F : écart max = {ecart:.1f}%")

# Vérifier que les % par âge font ~100%
for df, nom in [(df_atmp, "AT/MP"), (df_insee, "INSEE")]:
    cols_age = [c for c in df.columns if any(x in c for x in ['18_29', '30_44', '45_59', '60'])]
    total_age = df[cols_age].sum(axis=1)
    ecart = (total_age - 100).abs().max()
    status = "✅" if ecart < 1 else "⚠️"
    print(f"{status} {nom} — Total tranches d'âge : écart max = {ecart:.1f}%")

# Vérifier la clé de jointure CTN
ctns_atmp = set(df_atmp['ctn'])
ctns_sumer = set(df_sumer['ctn'])
ctns_insee = set(df_insee['ctn'])
print(f"\n{'✅' if ctns_atmp == ctns_sumer == ctns_insee else '⚠️'} "
      f"Clé de jointure CTN cohérente entre les 3 sources")
print(f"   CTN communs : {sorted(ctns_atmp & ctns_sumer & ctns_insee)}")


# 4. Visualisations exploratoires


## 4.1 Indice de fréquence AT par secteur


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
df_sorted = df_atmp.sort_values('indice_frequence', ascending=True)
bars = ax.barh(df_sorted['secteur'], df_sorted['indice_frequence'], color=PALETTE[:len(df_sorted)])
ax.set_xlabel("Indice de fréquence (AT pour 1 000 salariés)")
ax.set_title("Indice de fréquence des AT par secteur (CTN)")
ax.axvline(x=df_atmp['indice_frequence'].mean(), color='red', linestyle='--',
           alpha=0.7, label=f"Moyenne = {df_atmp['indice_frequence'].mean():.1f}")
ax.legend()
for bar, val in zip(bars, df_sorted['indice_frequence']):
    ax.text(val + 0.5, bar.get_y() + bar.get_height()/2, f'{val:.1f}',
            va='center', fontsize=10)
plt.tight_layout()
plt.savefig('../assets/01_indice_frequence_at.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure sauvegardée : assets/01_indice_frequence_at.png")


## 4.2 Répartition H/F dans les données AT/MP vs INSEE


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Données AT/MP
x = np.arange(len(df_atmp))
width = 0.35
axes[0].bar(x - width/2, df_atmp['pct_hommes'], width, label='Hommes', color='#185FA5')
axes[0].bar(x + width/2, df_atmp['pct_femmes'], width, label='Femmes', color='#D85A30')
axes[0].set_xticks(x)
axes[0].set_xticklabels([f"CTN {c}" for c in df_atmp['ctn']], rotation=45)
axes[0].set_ylabel('%')
axes[0].set_title('Répartition H/F dans les AT/MP')
axes[0].legend()
axes[0].set_ylim(0, 100)

# Données INSEE
axes[1].bar(x - width/2, df_insee['pct_hommes_insee'], width, label='Hommes', color='#185FA5', alpha=0.6)
axes[1].bar(x + width/2, df_insee['pct_femmes_insee'], width, label='Femmes', color='#D85A30', alpha=0.6)
axes[1].set_xticks(x)
axes[1].set_xticklabels([f"CTN {c}" for c in df_insee['ctn']], rotation=45)
axes[1].set_ylabel('%')
axes[1].set_title('Répartition H/F population active (INSEE)')
axes[1].legend()
axes[1].set_ylim(0, 100)

plt.suptitle("Comparaison H/F : données AT/MP vs population active réelle", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../assets/01_comparaison_hf.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure sauvegardée : assets/01_comparaison_hf.png")


## 4.3 Heatmap des expositions SUMER par secteur


In [ ]:
cols_expo = [c for c in df_sumer.columns if c.startswith('pct_')]
labels_expo = [c.replace('pct_', '').replace('_', ' ').title() for c in cols_expo]

fig, ax = plt.subplots(figsize=(14, 7))
data_heatmap = df_sumer.set_index('secteur')[cols_expo]
data_heatmap.columns = labels_expo

sns.heatmap(data_heatmap, annot=True, fmt='.0f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, cbar_kws={'label': '% de salariés exposés'})
ax.set_title("Expositions aux risques professionnels par secteur (SUMER 2017)")
ax.set_ylabel('')
plt.tight_layout()
plt.savefig('../assets/01_heatmap_expositions.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure sauvegardée : assets/01_heatmap_expositions.png")


## 4.4 Couverture des données AT/MP vs population active


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

df_merge = df_atmp[['ctn', 'secteur', 'effectifs_salaries']].merge(
    df_insee[['ctn', 'population_active']], on='ctn'
)
df_merge['taux_couverture'] = (df_merge['effectifs_salaries'] / df_merge['population_active'] * 100)

bars = ax.barh(df_merge['secteur'],
               df_merge['taux_couverture'],
               color=['#0F6E56' if v > 90 else '#BA7517' if v > 80 else '#E24B4A'
                      for v in df_merge['taux_couverture']])
ax.set_xlabel("Taux de couverture (%)")
ax.set_title("Couverture des données AT/MP par rapport à la population active (INSEE)")
ax.axvline(x=100, color='gray', linestyle=':', alpha=0.5)
for bar, val in zip(bars, df_merge['taux_couverture']):
    ax.text(val + 0.5, bar.get_y() + bar.get_height()/2, f'{val:.0f}%',
            va='center', fontsize=10)
plt.tight_layout()
plt.savefig('../assets/01_couverture_donnees.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure sauvegardée : assets/01_couverture_donnees.png")


# 5. Synthèse du profilage

**Constats clés :**

1. **9 secteurs CTN** couverts dans les 3 sources — clé de jointure cohérente
2. **Aucune valeur manquante** détectée
3. **Disparité de couverture** : les effectifs AT/MP ne représentent pas
   100% de la population active de chaque secteur (régime général uniquement)
4. **Forte hétérogénéité** des indices de fréquence entre secteurs
   (de 10 à 65 AT/1000 salariés)
5. **Les données sont prêtes** pour l'analyse de biais (Notebook 02)

**Prochaine étape** : Notebook 02 — Analyse de représentativité et détection
des biais démographiques (âge, sexe) dans les données AT/MP par rapport à
la population active réelle.


In [ ]:
print("="*60)
print("  NOTEBOOK 01 — EXPLORATION TERMINÉE")
print("="*60)
print(f"\n  Datasets chargés : 3")
print(f"  Secteurs CTN     : {len(df_atmp)}")
print(f"  Figures générées  : 4")
print(f"\n  Prochaine étape   : 02_biais.ipynb")
print(f"  → Analyse de représentativité et détection des biais")
